In [1]:
import chromadb
import pandas as pd
import os

# 1. Configuration
db_path = os.path.abspath(r"D:\pY\InsuranceRAG\local_db")
print(f"📂 Looking for database at: {db_path}")

client = chromadb.PersistentClient(path=db_path)
collections = client.list_collections()
collection_names = [c.name for c in collections]
print(f"📚 Found {len(collections)} collections: {collection_names}")

# 2. Function to inspect a specific collection
def inspect_collection_dynamic(name):
    if name not in collection_names:
        print(f"❌ Collection '{name}' does not exist.")
        return None
    
    collection = client.get_collection(name=name)
    results = collection.get(include=["documents", "metadatas"])
    
    if not results['ids']:
        print(f"⚠️ Collection '{name}' exists but is empty.")
        return None
        
    records = []
    for i in range(len(results['ids'])):
        # 1. Start with the core fields (ID and Document Preview)
        entry = {
            "ID": results['ids'][i],
            "Content Preview": results['documents'][i][:100].replace('\n', ' ') + "..."
        }
        
        # 2. Dynamically add all metadata fields
        meta = results['metadatas'][i] if results['metadatas'] else {}
        for key, value in meta.items():
            # If the value is a file path, just show the filename for neatness
            if key == "source" and isinstance(value, str):
                entry["Source File"] = os.path.basename(value)
            else:
                entry[key] = value
                
        records.append(entry)
    
    # 3. Create DataFrame (Pandas handles the alignment of different keys automatically)
    df = pd.DataFrame(records)
    
    # Optional: Move 'ID' and 'Content Preview' to the end for better readability
    cols = [c for c in df.columns if c not in ["ID", "Content Preview"]] + ["ID", "Content Preview"]
    return df[cols]

# 3. Fetch and Display Both Collections
print("\n--- 📜 MASTER POLICIES ---")
df_policies = inspect_collection_dynamic("policy_master_collection")
if df_policies is not None:
    display(df_policies.sort_values(by="Source File"))

print("\n--- 📑 CLIENT CLAIMS ---")
df_claims = inspect_collection_dynamic("claims_collection")
if df_claims is not None:
    display(df_claims)

print("\n--- 📑 EVALUATION AUDIT LOG ---")
df_audit = inspect_collection_dynamic("evaluation_audit_log")
if df_audit is not None:
    display(df_audit)

print("\n--- 📑 INSTANCE LOG ---")
df_instance_log = inspect_collection_dynamic("instance_log")
if df_instance_log is not None:
    display(df_instance_log)

# 4. Summary Table
if df_policies is not None or df_claims is not None:
    print("\n📊 Database Summary:")
    for c in collections:
        print(f" - {c.name}: {c.count()} chunks")

📂 Looking for database at: D:\pY\InsuranceRAG\local_db
📚 Found 4 collections: ['claims_collection', 'evaluation_audit_log', 'policy_master_collection', 'instance_log']

--- 📜 MASTER POLICIES ---
⚠️ Collection 'policy_master_collection' exists but is empty.

--- 📑 CLIENT CLAIMS ---
⚠️ Collection 'claims_collection' exists but is empty.

--- 📑 EVALUATION AUDIT LOG ---
⚠️ Collection 'evaluation_audit_log' exists but is empty.

--- 📑 INSTANCE LOG ---
⚠️ Collection 'instance_log' exists but is empty.
